# 03 multicentres calcul

Protocole de calcul conservé pour la traçabilité. Les sorties exécutées sont dans les notebooks de synthèse. Les données et modèles restent dans le dossier `data` du projet initial. Une nouvelle expérience après consultation du test est exploratoire. Aucun calcul n’est lancé par la lecture de ce fichier.

**Version commentée.** Lire les sections dans l’ordre. Les petites fonctions sont définies une seule fois, puis utilisées plus bas. Les calculs restent désactivés par défaut. Voir `../../docs/CHOIX_ET_JUSTIFICATIONS.md`.


In [ ]:
EXECUTER_CALCULS = False
if not EXECUTER_CALCULS:
    raise RuntimeError('Calculs désactivés. Consulter les notebooks de synthèse. Lire le README de cette contribution avant toute relance.')



# CAMELYON17 — Comparaison des méthodes entre établissements

## Objectif et périmètre
Comparer les images originales, Reinhard et Macenko avec une extraction identique et un modèle ResNet18 préentraîné utilisé comme extracteur fixe, suivi d'une régression logistique. Les centres 0 et 4 alimentent l'entraînement et la validation ; le centre 2 est réservé au test.

Cette étude exploratoire utilise 14 lames et des quotas de patchs. Les scores ne représentent pas la prévalence tumorale dans les lames entières. Les quatre lames de test appartiennent à quatre identifiants patients différents : elles ne permettent pas de calculer un stade pN complet. Les cartes de chaleur exhaustives, les classes de lames et le pN restent à recontrôler dans la chaîne commune.

**Exécution :** placer ce notebook dans `notebooks`, choisir le noyau `pt`, puis exécuter dans l'ordre. Les résultats sont enregistrés sous `data/multi_centres/experiences`. Les sorties ont été retirées de cette copie de reproduction ; les synthèses conservent les résultats exécutés.

Dépendances supplémentaires, à installer dans le terminal :
```powershell
& "C:\Users\Elodi\pt\Scripts\python.exe" -m pip install torchstain
```
Les autres bibliothèques sont celles du prototype et d'OpenSlide.

**Lecture du code.** Les bibliothèques ont quatre rôles : lire les images, manipuler des tableaux, appliquer le modèle et afficher les résultats.

In [ ]:
from pathlib import Path
import sys, json, hashlib, time, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import openslide, torch, torchvision, torchstain, joblib
from torchvision import transforms
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from IPython.display import display
PROJET = next((p for p in [Path.cwd(),
    *Path.cwd().parents] if (p / 'data/multi_centres/inventaire/selection_multicentres.csv').exists()),
    None)
assert PROJET is not None, 'Sélection introuvable : ouvrir le projet camelyon17-main.'
BASE = PROJET / 'data/multi_centres'
print('Python :', sys.executable)
print('Projet :', PROJET)



## 1. Paramètres et reproductibilité
Le niveau 1 produit des images proches de 0,5 µm/pixel, avec une différence physique entre centres conservée dans les métadonnées. Les niveaux du masque sont vérifiés indépendamment.

Un patch tumoral contient au moins un pixel tumoral dans son carré central de 112 pixels. Un patch négatif ne contient aucun pixel tumoral dans toute l'image. Les patchs présentant uniquement une tumeur périphérique sont écartés. Le tissu est filtré par saturation HSV ≥ 0,04 sur au moins 25 % des pixels. Les masques réduits servent uniquement à proposer des candidats ; les labels sont vérifiés au niveau de lecture.

Les quotas limitent le coût, sans garantir un effectif identique si une petite lésion fournit peu de patchs. Aucun patch n'est dupliqué pour atteindre un quota. Chaque session de calcul crée un dossier daté distinct, même avec les mêmes paramètres.

**Lecture du code.** CONFIG contient les réglages d’extraction. centre signifie ici la largeur du carré central (112 pixels), alors que la colonne centre des tableaux désigne l’hôpital (0, 2 ou 4). Le dossier d’une expérience déjà figée est protégé.

In [ ]:
CONFIG = dict(version='multicentres_v2_masque_valide',
    seed=42,
    niveau=1,
    taille=224,
    centre=112,
    tissu_min=0.25,
    saturation_min=0.04,
    positifs_par_lame=200,
    negatifs_lame_positive=100,
    negatifs_lame_negative=200)
identifiant = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
# Une nouvelle sortie conserve les données brutes et protège l’expérience publiée.
from datetime import datetime
session = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
EXP = PROJET / "data/reproductions_multicentres" / (identifiant + "_" + session)
if (EXP / 'choix_final.json').exists():
    raise RuntimeError('Expérience figée : reproduction interdite dans ce dossier. Utiliser une copie séparée du projet.')
EXP.mkdir(parents=True, exist_ok=True)
(EXP / 'configuration.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
versions = {p: importlib.metadata.version(p) for p in ['numpy',
    'pandas',
    'torch',
    'torchvision',
    'torchstain',
    'scikit-learn',
    'openslide-python']}
(EXP / 'environnement.json').write_text(json.dumps(dict(python=sys.version,
    plateforme=platform.platform(),
    versions=versions),
    indent=2),
    encoding='utf-8')
selection = pd.read_csv(BASE / 'inventaire/selection_multicentres.csv')
assert selection.groupby('patiente').ensemble.nunique().max() == 1
assert set(selection.loc[selection.ensemble != 'test', 'centre']) == {0, 4}
assert set(selection.loc[selection.ensemble == 'test', 'centre']) == {2}
selection.to_csv(EXP / 'selection.csv', index=False)
display(selection[['lame', 'patiente', 'centre', 'classe', 'ensemble']])
print('Résultats :', EXP)



## 2. Contrôle visuel des lames et des masques
La superposition rouge représente le masque tumoral fourni, et non une prédiction. Une vue globale peut masquer les micrométastases ; les vues locales de la section suivante complètent ce contrôle.

**Lecture du code.** Le rouge représente les annotations de référence. Le niveau 6 sert seulement à afficher une vue globale, pas à extraire les patchs du modèle.

In [ ]:
lames_controle = selection[(selection.ensemble == 'train') & (selection.classe != 'negative')]
fig, axes = plt.subplots(len(lames_controle), 2, figsize=(11, 4 * len(lames_controle)), squeeze=False)
for k, r in enumerate(lames_controle.itertuples()):
    with openslide.OpenSlide(str(BASE / 'raw/images' / (r.lame + '.tif'))) as im, openslide.OpenSlide(str(BASE / 'raw/masks' / (r.lame + '_mask.tif'))) as ma:
        assert im.dimensions == ma.dimensions
        lev = 6
        assert im.level_dimensions[lev] == ma.level_dimensions[lev]
        rgb = np.array(im.read_region((0, 0), lev, im.level_dimensions[lev]).convert('RGB'))
        lab = np.array(ma.read_region((0, 0), lev, ma.level_dimensions[lev]))[:, :, 0]
        assert np.isin(lab, [0, 1, 2]).all()
        overlay = np.zeros((*lab.shape, 4), dtype=np.float32)
        overlay[lab == 2] = [1, 0, 0, 0.6]
        axes[k, 0].imshow(rgb)
        axes[k, 1].imshow(rgb)
        axes[k, 1].imshow(overlay)
        axes[k, 0].set_title(f'Centre {r.centre} — {r.lame}')
        axes[k, 1].set_title('Masque tumoral')
        for ax in axes[k]:
            ax.axis('off')
fig.tight_layout()
fig.savefig(EXP / '01_lames_masques.png', dpi=120)
plt.show()



## 3. Extraction reproductible des patchs
L'extraction se fait sur une grille sans chevauchement, puis les candidats sont parcourus dans un ordre pseudo-aléatoire fixé par lame. Le masque réduit peut ne pas conserver les foyers les plus petits : cette limite concerne notamment les cellules isolées, exclues de cette sélection.

Les extractions terminées peuvent être réutilisées au sein de la même session de calcul. Une extraction interrompue est recalculée pour la lame concernée. Ne pas réexécuter la configuration pour reprendre la même session : cela crée un nouveau dossier. Les durées enregistrées correspondent à l'extraction initiale.

Les régions dont le masque ou l’image contient des pixels transparents sont exclues et comptabilisées dans `bilan.json`. Les pixels non renseignés ne sont pas assimilés au tissu sain. Cette exclusion peut modifier les effectifs et la représentativité ; vérifier les bilans par lame. La version v2 utilise un dossier distinct pour ne pas mélanger les extractions.

**Lecture du code.** Trois fonctions : lire les labels, décider de la classe, puis parcourir une lame. Dans le masque, la valeur 2 signifie tumeur ; dans le tableau final y_vrai, la valeur 1 signifie tumoral. OpenSlide attend les coordonnées x et y dans le repère du niveau 0.

In [ ]:
# Lire les annotations ; renvoyer None si la région contient des pixels non renseignés.
def lire_labels(ma, xy, niveau, taille):
    a = np.array(ma.read_region(xy, niveau, (taille, taille)))
    if a.ndim != 3 or a.shape[2] != 4:
        raise ValueError('Le masque lu doit être au format RGBA.')
    if not (a[:, :, 3] == 255).all():
        return None
    lab = a[:, :, 0]
    assert np.isin(lab, [0, 1, 2]).all(), 'Valeurs du masque inattendues.'
    return lab



In [ ]:
# Attribuer 1 si tumeur au centre, 0 si aucune tumeur, sinon exclure avec None.
def classe_patch(labels):
    n = labels.shape[0]
    c = CONFIG['centre']
    debut = (n - c) // 2
    if (labels[debut:debut + c, debut:debut + c] == 2).any():
        return 1
    if not (labels == 2).any():
        return 0
    return None



In [ ]:
# Extraire des patchs valides d’une lame et conserver leurs coordonnées et labels.
def extraire_lame(r):
    dossier = EXP / 'patchs' / r.lame
    dossier.mkdir(parents=True, exist_ok=True)
    csv = dossier / 'metadata.csv'
    if csv.exists():
        df = pd.read_csv(csv)
        if len(df) and all(((EXP / p).exists() for p in df.chemin)):
            return df
    debut = time.perf_counter()
    rows = []
    seed = CONFIG['seed'] + int(hashlib.sha256(r.lame.encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    n = CONFIG['taille']
    niveau = CONFIG['niveau']
    quotas = {0: CONFIG['negatifs_lame_negative'] if r.classe == 'negative' else CONFIG['negatifs_lame_positive'],
        1: 0 if r.classe == 'negative' else CONFIG['positifs_par_lame']}
    comptes = {0: 0, 1: 0}
    controles = dict(candidats_lus=0, masque_incomplet=0, image_incomplete=0)
    print('Extraction :', r.lame, flush=True)
    with openslide.OpenSlide(str(BASE / 'raw/images' / (r.lame + '.tif'))) as im, openslide.OpenSlide(str(BASE / 'raw/masks' / (r.lame + '_mask.tif'))) as ma:
        assert im.dimensions == ma.dimensions
        # Facteur entre le niveau lu et le niveau 0 des coordonnées.
        ds = im.level_downsamples[niveau]
        ml = int(np.argmin(np.abs(np.array(ma.level_downsamples) - ds)))
        assert abs(ma.level_downsamples[ml] - ds) < 0.001
        grossier = min(5, ma.level_count - 1)
        dg = ma.level_downsamples[grossier]
        low_rgba = np.array(ma.read_region((0, 0), grossier, ma.level_dimensions[grossier]))
        low = low_rgba[:, :, 0].copy()
        # Les zones transparentes sont exclues des candidats, pas déclarées saines.
        low[low_rgba[:, :, 3] != 255] = 0
        assert np.isin(low, [0, 1, 2]).all()
        candidats = {0: [], 1: []}
        largeur, hauteur = im.level_dimensions[niveau]
        for yy in range(0, hauteur - n + 1, n):
            for xx in range(0, largeur - n + 1, n):
                x, y = (round(xx * ds), round(yy * ds))
                if x + n * ds > im.dimensions[0] or y + n * ds > im.dimensions[1]:
                    continue
                a, b = (int(x / dg), int(y / dg))
                z = low[b:int(np.ceil((y + n * ds) / dg)), a:int(np.ceil((x + n * ds) / dg))]
                if z.size and (z > 0).any():
                    candidats[int((z == 2).any())].append((x, y))
        for groupe in [1, 0]:
            rng.shuffle(candidats[groupe])
            for xy in candidats[groupe]:
                if all((comptes[k] >= quotas[k] for k in [0, 1])):
                    break
                if groupe == 0 and comptes[0] >= quotas[0]:
                    break
                controles['candidats_lus'] += 1
                lab = lire_labels(ma, xy, ml, n)
                if lab is None:
                    controles['masque_incomplet'] += 1
                    continue
                classe = classe_patch(lab)
                if classe is None or comptes[classe] >= quotas[classe]:
                    continue
                region = im.read_region(xy, niveau, (n, n))
                if not (np.asarray(region)[:, :, 3] == 255).all():
                    controles['image_incomplete'] += 1
                    continue
                image = region.convert('RGB')
                sat = np.asarray(image.convert('HSV'))[:, :, 1] / 255
                tissu = float((sat >= CONFIG['saturation_min']).mean())
                if tissu < CONFIG['tissu_min']:
                    continue
                nom = f'x{xy[0]}_y{xy[1]}.png'
                image.save(dossier / nom)
                rows.append(dict(lame=r.lame,
                    patiente=r.patiente,
                    centre=r.centre,
                    ensemble=r.ensemble,
                    classe_lame=r.classe,
                    y_vrai=classe,
                    x=xy[0],
                    y=xy[1],
                    niveau=niveau,
                    mpp_x=float(im.properties['openslide.mpp-x']) * ds,
                    pct_tissu=tissu,
                    pct_tumeur=float((lab == 2).mean()),
                    chemin=(dossier / nom).relative_to(EXP).as_posix()))
                comptes[classe] += 1
    if not rows:
        (dossier / 'diagnostic_echec.json').write_text(json.dumps(controles, indent=2), encoding='utf-8')
        raise RuntimeError(f'Aucun patch valide pour {r.lame}. Diagnostic : {controles}. Examiner le masque avant de poursuivre.')
    df = pd.DataFrame(rows)
    df.to_csv(csv, index=False)
    (dossier / 'bilan.json').write_text(json.dumps(dict(secondes=time.perf_counter() - debut,
        quotas=quotas,
        obtenus=comptes,
        controles=controles),
        indent=2),
        encoding='utf-8')
    print(r.lame, 'patchs :', comptes, 'contrôles :', controles, flush=True)
    return df



In [ ]:
patchs = pd.concat([extraire_lame(r) for r in selection[selection.ensemble != 'test'].itertuples()],
    ignore_index=True)
patchs.to_csv(EXP / 'patchs_developpement.csv', index=False)
display(patchs.groupby(['ensemble', 'centre', 'lame', 'y_vrai']).size().rename('effectif').to_frame())
assert patchs.groupby('patiente').ensemble.nunique().max() == 1
assert set(patchs.y_vrai) == {0, 1}



**Lecture du code.** Afficher un exemple par ensemble, centre et classe permet de vérifier visuellement les images et leurs labels.

In [ ]:
exemples = patchs.groupby(['ensemble', 'centre', 'y_vrai'], group_keys=False).head(1)
fig, axes = plt.subplots(len(exemples), 2, figsize=(8, 3 * len(exemples)), squeeze=False)
for k, r in enumerate(exemples.itertuples()):
    rgb = np.array(Image.open(EXP / r.chemin).convert('RGB'))
    with openslide.OpenSlide(str(BASE / 'raw/masks' / (r.lame + '_mask.tif'))) as ma:
        lab = lire_labels(ma, (r.x, r.y), CONFIG['niveau'], CONFIG['taille'])
    if lab is None:
        raise RuntimeError('Masque incomplet dans un patch enregistré : ' + r.lame)
    axes[k, 0].imshow(rgb)
    axes[k, 1].imshow(rgb)
    ov = np.zeros((*lab.shape, 4))
    ov[lab == 2] = [1, 0, 0, 0.5]
    axes[k, 1].imshow(ov)
    d = (CONFIG['taille'] - CONFIG['centre']) // 2
    axes[k,
        1].add_patch(plt.Rectangle((d,
        d),
        CONFIG['centre'],
        CONFIG['centre'],
        fill=False,
        edgecolor='cyan'))
    axes[k, 0].set_title(f'{r.ensemble} / centre {r.centre} / label {r.y_vrai}')
    axes[k, 1].set_title(r.lame)
    for ax in axes[k]:
        ax.axis('off')
fig.tight_layout()
fig.savefig(EXP / '02_patchs_masques.png', dpi=120)
plt.show()



## 4. Référence de normalisation issue de l'entraînement
Une mosaïque de 16 patchs d'entraînement, répartis entre les deux centres et les deux labels, sert de référence commune. Elle est figée avant l'évaluation. Reinhard ajuste les statistiques de couleur ; Macenko estime une représentation des colorations en densité optique. Les implémentations utilisées sont celles de `torchstain`, backend NumPy.

En cas d'échec numérique sur un patch, l'image originale est conservée et l'échec est journalisé : les trois conditions gardent exactement les mêmes exemples. La fréquence de ces replis fait partie du bilan de chaque méthode.

**Lecture du code.** La référence utilise seulement l’entraînement. Reinhard et Macenko sont appelés via torchstain : leurs algorithmes ne sont pas réécrits ici. Un échec est conservé dans un journal.

In [ ]:
train = patchs[patchs.ensemble == 'train']
refs = []
for _, g in train.groupby(['centre', 'y_vrai']):
    if len(g) < 4:
        raise ValueError('Moins de 4 patchs dans un groupe de référence.')
    refs.extend(g.sample(4, random_state=CONFIG['seed']).chemin.tolist())
assert len(refs) == 16
reference = np.zeros((4 * 224, 4 * 224, 3), dtype=np.uint8)
for k, p in enumerate(refs):
    reference[k // 4 * 224:(k // 4 + 1) * 224,
        k % 4 * 224:(k % 4 + 1) * 224] = np.array(Image.open(EXP / p).convert('RGB'))
Image.fromarray(reference).save(EXP / 'reference_normalisation.png')
(EXP / 'reference_patchs.json').write_text(json.dumps(refs, indent=2), encoding='utf-8')
normaliseurs = {'reinhard': torchstain.normalizers.ReinhardNormalizer(backend='numpy'),
    'macenko': torchstain.normalizers.MacenkoNormalizer(backend='numpy')}
for normaliseur in normaliseurs.values():
    normaliseur.fit(reference)
METHODES = ['original', 'reinhard', 'macenko']



In [ ]:
# Appliquer une méthode de couleur ; conserver l’original et signaler tout échec.
def normaliser(rgb, methode):
    if methode == 'original':
        return (rgb, '')
    try:
        with np.errstate(all='raise'):
            z = normaliseurs[methode].normalize(I=rgb)
        if isinstance(z, tuple):
            z = z[0]
        z = np.asarray(z)
        if z.shape != rgb.shape or not np.isfinite(z).all():
            raise ValueError('Sortie invalide')
        return (np.clip(z, 0, 255).astype(np.uint8), '')
    except Exception as e:
        return (rgb, type(e).__name__ + ': ' + str(e))



In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(10, 12))
for i, (_, g) in enumerate(train.groupby(['centre', 'y_vrai'])):
    r = g.iloc[0]
    rgb = np.array(Image.open(EXP / r.chemin).convert('RGB'))
    for j, m in enumerate(METHODES):
        z, err = normaliser(rgb, m)
        axes[i, j].imshow(z)
        axes[i, j].axis('off')
        axes[i, j].set_title(f'{m} — centre {r.centre}, label {r.y_vrai}' + (' (repli)' if err else ''))
fig.tight_layout()
fig.savefig(EXP / '03_normalisations.png', dpi=150)
plt.show()



## 5. Caractéristiques ResNet18 et mesure des temps
Les poids ImageNet sont téléchargés au premier lancement si nécessaire. Le CNN reste fixe. Les images sont normalisées avec les constantes ImageNet après la transformation colorimétrique. Les temps distinguent normalisation et extraction des caractéristiques, hors lecture disque et hors téléchargement des poids. Les mêmes tailles de lots et le même appareil sont utilisés pour toutes les méthodes.

**Lecture du code.** Le réseau reste fixe. Le tableau X contient une ligne par patch et 512 colonnes. Les lots de 32 évitent de charger tous les patchs simultanément. Les temps ne couvrent pas toute la chaîne.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(CONFIG['seed'])
torch.set_num_threads(min(4, torch.get_num_threads()))
reseau = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
reseau.fc = torch.nn.Identity()
reseau.eval().to(DEVICE)
preparation = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize([0.485,
    0.456,
    0.406],
    [0.229,
    0.224,
    0.225])])
print('Appareil :', DEVICE)
with torch.inference_mode():
    reseau(torch.zeros(1, 3, 224, 224, device=DEVICE))
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()



In [ ]:
# Produire les 512 caractéristiques de chaque patch et mesurer les durées.
def caracteristiques(df, methode, nom):
    fichiers = []
    temps = []
    erreurs = []
    for debut in range(0, len(df), 32):
        lot = df.iloc[debut:debut + 32]
        tenseurs = []
        tn = {}
        for r in lot.itertuples():
            rgb = np.array(Image.open(EXP / r.chemin).convert('RGB'))
            t = time.perf_counter()
            z, err = normaliser(rgb, methode)
            tn[r.centre] = tn.get(r.centre, 0) + time.perf_counter() - t
            if err:
                erreurs.append(dict(chemin=r.chemin, centre=r.centre, erreur=err))
            tenseurs.append(preparation(Image.fromarray(z)))
        x = torch.stack(tenseurs).to(DEVICE)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        t = time.perf_counter()
        with torch.inference_mode():
            features = reseau(x).cpu().numpy()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        duree = time.perf_counter() - t
        fichiers.append(features)
        for centre, nb in lot.centre.value_counts().items():
            temps.append(dict(centre=int(centre),
                n=int(nb),
                normalisation_s=tn[centre],
                cnn_s=duree * nb / len(lot)))
        if debut % 320 == 0:
            print(nom, methode, min(debut + 32, len(df)), '/', len(df), flush=True)
    pd.DataFrame(erreurs,
        columns=['chemin',
        'centre',
        'erreur']).to_csv(EXP / f'erreurs_{nom}_{methode}.csv',
        index=False)
    tdf = pd.DataFrame(temps).groupby('centre').sum().reset_index()
    tdf.to_csv(EXP / f'temps_{nom}_{methode}.csv', index=False)
    return (np.concatenate(fichiers), tdf, len(erreurs))



## 6. Entraînement et choix sur la validation
Une régression logistique L2 est ajustée pour chaque méthode et chaque valeur de C parmi 0,001 ; 0,01 ; 0,1 ; 1 ; 10. La standardisation est ajustée uniquement sur l'entraînement.

La valeur de C maximise l'AUC moyenne des deux centres de validation. Le seuil commun aux deux centres est le minimum des seuils maximaux permettant d'atteindre 95 % de rappel dans chacun. Il reste unique pour une méthode et sera conservé pour le test. La méthode est proposée selon sa spécificité moyenne par centre, puis son AUC moyenne, parmi celles atteignant 95 % de rappel dans chacun des deux centres. Si aucune n'y parvient, aucune méthode n'est figée automatiquement.

Cette règle constitue un choix expérimental, pas une garantie clinique. La validation sert à plusieurs sélections ; ses scores peuvent donc être optimistes.

La section 6 bis conserve le recalcul effectué sur les scores de validation. Les sorties historiques sont consultables dans la synthèse principale.

**Lecture du code.** Pour chaque méthode : calculer X, essayer cinq valeurs de C, retenir l’AUC moyenne la plus élevée sur la validation, puis choisir un seuil commun. Une valeur de C plus petite impose une régularisation plus forte. Le test n’intervient jamais dans ce choix.

In [ ]:
if (EXP / 'choix_final.json').exists():
    raise RuntimeError('Le modèle figé ne doit pas être réentraîné ni remplacé.')



In [ ]:
# Calculer VP, VN, FP, FN, rappel, précision, spécificité et AUC.
def metriques(y, score, seuil):
    y = np.asarray(y)
    score = np.asarray(score)
    pred = score >= seuil
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n=len(y),
        VP=int(vp),
        VN=int(vn),
        FP=int(fp),
        FN=int(fn),
        rappel=vp / (vp + fn) if vp + fn else np.nan,
        precision=vp / (vp + fp) if vp + fp else np.nan,
        specificite=vn / (vn + fp) if vn + fp else np.nan,
        auc=roc_auc_score(y,
        score) if len(np.unique(y)) == 2 else np.nan)



In [ ]:
# Choisir le plus grand seuil qui conserve au moins 95 % des positifs du centre.
def choisir_seuil(y, p, cible=0.95):
    # Trier uniquement les scores des vrais positifs de référence.
    positifs = np.sort(np.asarray(p)[np.asarray(y) == 1])
    if not len(positifs):
        raise ValueError('Aucun positif en validation')
    return float(positifs[len(positifs) - int(np.ceil(cible * len(positifs)))])



In [ ]:
resultats = []
recherche = []
modeles = {}
propositions = []
train_idx = (patchs.ensemble == 'train').to_numpy()
val_idx = ~train_idx
val = patchs.loc[val_idx].reset_index(drop=True)
for methode in METHODES:
    X, temps, n_erreurs = caracteristiques(patchs, methode, 'developpement')
    meilleur = None
    for C in [0.001, 0.01, 0.1, 1, 10]:
        modele = make_pipeline(StandardScaler(),
            LogisticRegression(C=C,
            solver='lbfgs',
            max_iter=3000,
            random_state=42))
        t = time.perf_counter()
        # Apprendre uniquement sur les patchs d’entraînement.
        modele.fit(X[train_idx], patchs.loc[train_idx, 'y_vrai'])
        fit_s = time.perf_counter() - t
        p = modele.predict_proba(X[val_idx])[:, 1]
        aucs = [roc_auc_score(val.loc[val.centre == c,
            'y_vrai'],
            p[val.centre == c]) for c in sorted(val.centre.unique())]
        moyenne = float(np.mean(aucs))
        recherche.append(dict(methode=methode, C=C, auc_moyenne=moyenne, fit_s=fit_s))
        if meilleur is None or moyenne > meilleur[0]:
            meilleur = (moyenne, C, modele, p, fit_s)
    auc, C, modele, p, fit_s = meilleur
    seuil = min((choisir_seuil(val.loc[val.centre == c,
        'y_vrai'],
        p[val.centre == c]) for c in sorted(val.centre.unique())))
    predictions = val.copy()
    predictions['score'] = p
    predictions['prediction'] = (p >= seuil).astype(int)
    predictions.to_csv(EXP / f'predictions_validation_{methode}.csv', index=False)
    locaux = []
    for c, g in predictions.groupby('centre'):
        m = metriques(g.y_vrai, g.score, seuil)
        locaux.append(m)
        resultats.append(dict(methode=methode, centre=int(c), C=C, seuil=seuil, **m))
    propositions.append(dict(methode=methode,
        C=C,
        seuil=seuil,
        auc_moyenne=auc,
        rappel_min=min((m['rappel'] for m in locaux)),
        specificite_moyenne=np.mean([m['specificite'] for m in locaux]),
        replis=n_erreurs,
        fit_s=fit_s))
    modeles[methode] = modele
    joblib.dump(modele, EXP / f'modele_{methode}.joblib')
resultats = pd.DataFrame(resultats)
comparaison = pd.DataFrame(propositions)
resultats.to_csv(EXP / 'metriques_validation_par_centre.csv', index=False)
comparaison.to_csv(EXP / 'comparaison_methodes.csv', index=False)
pd.DataFrame(recherche).to_csv(EXP / 'recherche_C.csv', index=False)
display(resultats.round(4))
display(comparaison.round(4))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, met in zip(axes, ['rappel', 'precision', 'specificite']):
    resultats.pivot(index='methode',
        columns='centre',
        values=met).plot.bar(ax=ax,
        ylim=(0,
        1),
        title=met,
        rot=0)
fig.tight_layout()
fig.savefig(EXP / '04_validation_par_centre.png', dpi=150)
plt.show()
admissibles = comparaison[comparaison.rappel_min >= 0.95].sort_values(['specificite_moyenne',
    'auc_moyenne'],
    ascending=False)
print('Proposition selon la règle :',
    admissibles.iloc[0].methode if len(admissibles) else 'Aucune méthode admissible : analyser la validation.')



## 6 bis. Construire les exports de comparaison

Les premières expériences ont nécessité un recalibrage sur la validation pour respecter 95 % de rappel dans chaque centre. Cette version applique directement cette règle dans la section 6 et la réutilise ici pour préparer les exports détaillés. Le suffixe `_recalibre` est conservé pour rester compatible avec les résultats enregistrés. Aucun nouveau réglage n’est choisi sur le test.

**Lecture du code.** Cette étape fabrique les tableaux détaillés et les exports portant le suffixe _recalibre. Dans cette version, la même règle par centre est déjà appliquée pendant l’entraînement : ce n’est pas une deuxième optimisation. Les fonctions communes sont réutilisées.

In [ ]:
METHODES = ['original', 'reinhard', 'macenko']
initial = pd.read_csv(EXP / 'comparaison_methodes.csv').set_index('methode')
mesures = []
synthese = []
reference_identites = None
for methode in METHODES:
    df = pd.read_csv(EXP / f'predictions_validation_{methode}.csv')
    assert set(df.ensemble) == {'validation'} and set(df.centre) == {0, 4}
    assert np.isfinite(df.score).all() and df.score.between(0, 1).all()
    identites = df[['lame',
        'x',
        'y',
        'y_vrai',
        'centre']].sort_values(['lame',
        'x',
        'y']).reset_index(drop=True)
    assert not identites.duplicated(['lame', 'x', 'y']).any()
    if reference_identites is None:
        reference_identites = identites
    else:
        pd.testing.assert_frame_equal(reference_identites, identites)
    seuil = min((choisir_seuil(g.y_vrai, g.score) for _, g in df.groupby('centre')))
    locaux = []
    for centre, g in df.groupby('centre'):
        m = metriques(g.y_vrai, g.score, seuil)
        assert m['rappel'] >= 0.95
        locaux.append(m)
        mesures.append(dict(methode=methode, centre=int(centre), seuil=seuil, **m))
    df['prediction'] = (df.score >= seuil).astype(int)
    df.to_csv(EXP / f'predictions_validation_{methode}_recalibre.csv', index=False)
    temps = pd.read_csv(EXP / f'temps_developpement_{methode}.csv')
    synthese.append(dict(methode=methode,
        C=float(initial.loc[methode,
        'C']),
        seuil=seuil,
        rappel_min=min((m['rappel'] for m in locaux)),
        specificite_moyenne=float(np.mean([m['specificite'] for m in locaux])),
        auc_moyenne=float(np.mean([m['auc'] for m in locaux])),
        FP=sum((m['FP'] for m in locaux)),
        FN=sum((m['FN'] for m in locaux)),
        replis=int(initial.loc[methode,
        'replis']),
        normalisation_s=float(temps.normalisation_s.sum()),
        cnn_s=float(temps.cnn_s.sum()),
        temps_mesure_s=float(temps.normalisation_s.sum() + temps.cnn_s.sum())))
resultats = pd.DataFrame(mesures)
comparaison = pd.DataFrame(synthese)
admissibles = comparaison[comparaison.rappel_min >= 0.95].sort_values(['specificite_moyenne',
    'auc_moyenne'],
    ascending=False)
resultats.to_csv(EXP / 'metriques_validation_par_centre_recalibre.csv', index=False)
comparaison.to_csv(EXP / 'comparaison_methodes_recalibre.csv', index=False)
display(resultats.round(4))
display(comparaison.round(4))
print('Méthode proposée selon la règle :', admissibles.iloc[0].methode)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, met in zip(axes, ['rappel', 'precision', 'specificite']):
    resultats.pivot(index='methode',
        columns='centre',
        values=met).plot.bar(ax=ax,
        ylim=(0,
        1),
        title=met,
        rot=0)
fig.tight_layout()
fig.savefig(EXP / '04_validation_par_centre_recalibre.png', dpi=150)
plt.show()



## 7. Analyse des résultats de validation

La synthèse validée est présentée dans le notebook principal `03_multicentres.ipynb`. Les exports `_recalibre` sont la référence pour l’expérience enregistrée. Toute nouvelle exécution doit être analysée séparément.

## 8. Figer le choix avant le test
La cellule suivante ne modifie rien avec la valeur `False`. Après analyse des résultats et vérification des figures, passer `FIGER_CHOIX` à `True` pour enregistrer la proposition admissible. Un choix déjà enregistré n'est pas remplacé silencieusement.

Le test ne doit pas servir à essayer successivement les méthodes ou à modifier le seuil.

**Lecture du code.** Figer signifie enregistrer la méthode, C et le seuil choisis avant de regarder le test. False laisse le fichier inchangé ; True enregistre le choix si les contrôles passent.

In [ ]:
FIGER_CHOIX = False
choix_path = EXP / 'choix_final.json'
if FIGER_CHOIX:
    assert len(admissibles), 'Aucune méthode admissible selon la règle annoncée.'
    proposition = admissibles.iloc[0].to_dict()
    proposition['configuration'] = identifiant
    proposition['hash_reference'] = hashlib.sha256((EXP / 'reference_normalisation.png').read_bytes()).hexdigest()
    proposition['hash_modele'] = hashlib.sha256((EXP / f"modele_{proposition['methode']}.joblib").read_bytes()).hexdigest()
    if choix_path.exists():
        assert json.loads(choix_path.read_text()) == proposition, 'Un autre choix existe déjà : ne pas écraser un test antérieur.'
    else:
        choix_path.write_text(json.dumps(proposition, indent=2), encoding='utf-8')
    print('Choix figé :', proposition)
else:
    print('Choix non modifié. Test désactivé par défaut.')



In [ ]:
necessaires = ['extraire_lame',
    'caracteristiques',
    'normaliser',
    'reseau',
    'preparation',
    'DEVICE',
    'metriques',
    'selection',
    'EXP',
    'CONFIG',
    'choix_path',
    'identifiant']
manquants = [nom for nom in necessaires if nom not in globals()]
if manquants:
    print('Éléments à recharger :', manquants)
else:
    print('Tout est chargé : le test peut être lancé.')



## 9. Test du centre 2

Les commandes de téléchargement figurent dans `../../README.md`. Après avoir enregistré le choix, passer `EXECUTER_TEST` à `True` puis exécuter la cellule de test. Le centre 2 a déjà été observé : cette relance reproduit une expérience historique, elle ne crée pas une nouvelle réserve indépendante.

**Lecture du code.** Le test applique uniquement le choix figé. Il ne réentraîne rien. Les résultats existants sont relus au lieu d’être remplacés.

In [ ]:
EXECUTER_TEST = False
if EXECUTER_TEST:
    assert choix_path.exists(), 'Figer le choix sur la validation avant le test.'
    choix = json.loads(choix_path.read_text())
    methode = choix['methode']
    assert choix['configuration'] == identifiant
    assert hashlib.sha256((EXP / 'reference_normalisation.png').read_bytes()).hexdigest() == choix['hash_reference']
    modele_path = EXP / f'modele_{methode}.joblib'
    assert hashlib.sha256(modele_path.read_bytes()).hexdigest() == choix['hash_modele']
    resultat_test = EXP / 'metriques_test.csv'
    if resultat_test.exists():
        print('Résultats déjà enregistrés :')
        display(pd.read_csv(resultat_test))
    else:
        test = pd.concat([extraire_lame(r) for r in selection[selection.ensemble == 'test'].itertuples()],
            ignore_index=True)
        test.to_csv(EXP / 'patchs_test.csv', index=False)
        X, temps, n_erreurs = caracteristiques(test, methode, 'test')
        modele = joblib.load(modele_path)
        t = time.perf_counter()
        test['score'] = modele.predict_proba(X)[:, 1]
        prediction_s = time.perf_counter() - t
        test['prediction'] = (test.score >= choix['seuil']).astype(int)
        test.to_csv(EXP / 'predictions_test.csv', index=False)
        bilan = pd.DataFrame([dict(methode=methode,
            centre=2,
            seuil=choix['seuil'],
            replis=n_erreurs,
            regression_prediction_s=prediction_s,
            **metriques(test.y_vrai,
            test.score,
            choix['seuil']))])
        bilan.to_csv(resultat_test, index=False)
        display(bilan)
        par_lame = pd.DataFrame([dict(lame=nom,
            **metriques(g.y_vrai,
            g.score,
            choix['seuil'])) for nom,
            g in test.groupby('lame')])
        par_lame.to_csv(EXP / 'metriques_test_par_lame.csv', index=False)
        display(par_lame)
else:
    print('Test réservé non exécuté.')



**Lecture du code.** Cette dernière analyse décrit des erreurs déjà observées. Elle ne sert pas à ajuster le seuil. Le rouge est l’annotation et non une carte d’explication du modèle.

In [ ]:
# Décrire seulement le test de la session courante, s’il a été exécuté.
if EXECUTER_TEST and (EXP / "predictions_test.csv").exists():
    base = BASE
    experience = EXP
    configuration = json.loads((experience / 'configuration.json').read_text(encoding='utf-8'))
    choix = json.loads((experience / 'choix_final.json').read_text(encoding='utf-8'))
    predictions = pd.read_csv(experience / 'predictions_test.csv')
    lame = 'patient_046_node_4'
    erreurs = predictions.loc[(predictions.lame == lame) & (predictions.y_vrai == 1) & (predictions.prediction == 0)].sort_values(['score',
        'x',
        'y']).reset_index(drop=True)
    if erreurs.empty:
        raise ValueError('Aucun faux négatif trouvé pour cette lame.')
    indices = np.unique(np.linspace(0, len(erreurs) - 1, min(6, len(erreurs))).astype(int))
    exemples = erreurs.iloc[indices].copy()
    sortie = experience / 'analyse_erreurs_test'
    sortie.mkdir(exist_ok=True)
    exemples.to_csv(sortie / 'exemples_FN_micro.csv', index=False)
    fig, axes = plt.subplots(len(exemples), 2, figsize=(9, 3.3 * len(exemples)), squeeze=False)
    with openslide.OpenSlide(str(base / 'raw/images' / f'{lame}.tif')) as image_lame, openslide.OpenSlide(str(base / 'raw/masks' / f'{lame}_mask.tif')) as masque:
        for i, r in enumerate(exemples.itertuples()):
            with Image.open(experience / r.chemin) as fichier:
                rgb = np.array(fichier.convert('RGB'))
            hauteur, largeur = rgb.shape[:2]
            reduction = image_lame.level_downsamples[int(r.niveau)]
            niveau_masque = int(np.argmin(np.abs(np.array(masque.level_downsamples) - reduction)))
            if abs(masque.level_downsamples[niveau_masque] - reduction) >= 0.001:
                raise ValueError("Résolutions de l'image et du masque incompatibles.")
            region = np.array(masque.read_region((int(r.x), int(r.y)), niveau_masque, (largeur, hauteur)))
            if not (region[:, :, 3] == 255).all():
                raise ValueError('Masque incomplet pour un exemple sélectionné.')
            labels = region[:, :, 0]
            if not np.isin(labels, [0, 1, 2]).all():
                raise ValueError('Valeurs inattendues dans le masque.')
            superposition = np.zeros((hauteur, largeur, 4))
            superposition[labels == 2] = [1, 0, 0, 0.45]
            axes[i, 0].imshow(rgb)
            axes[i, 0].set_title(f'Faux négatif — score : {r.score:.3f}\nx={r.x}, y={r.y}')
            axes[i, 1].imshow(rgb)
            axes[i, 1].imshow(superposition)
            centre = int(configuration['centre'])
            axes[i,
                1].add_patch(Rectangle(((largeur - centre) // 2,
                (hauteur - centre) // 2),
                centre,
                centre,
                fill=False,
                edgecolor='cyan',
                linewidth=1.5))
            axes[i, 1].set_title(f'Annotation tumorale : {100 * (labels == 2).mean():.1f} % du patch')
            for ax in axes[i]:
                ax.axis('off')
    fig.suptitle(f"{lame} — exemples de patchs tumoraux manqués\nSeuil figé : {choix['seuil']:.3f} | Rouge : annotation ; cyan : zone centrale d'étiquetage",
        fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    figure = sortie / 'faux_negatifs_micro.png'
    fig.savefig(figure, dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure enregistrée :', figure)
    
else:
    print("Analyse des erreurs non exécutée : aucun test lancé dans cette session.")


### Résultat historique du 16 novembre — analyse des faux négatifs sur la lame avec micrométastase

Sur la lame patient_046_node_4, les 64 patchs étiquetés tumoraux
sont classés négatifs au seuil fixé sur la validation (0,314).
Leurs scores sont compris entre 0,010 et 0,192.

Six exemples ont été sélectionnés à différents niveaux de score.
Les zones annotées comme tumorales représentent entre 11,2 % et
32,7 % de leur surface. Elles sont localisées et parfois situées
en bordure de la zone centrale utilisée pour l’étiquetage.
Un patch est considéré comme tumoral dès qu’un pixel annoté
tumoral est présent dans cette zone centrale.

Ces observations suggèrent une difficulté possible à détecter
des foyers tumoraux minoritaires dans le patch. Cette hypothèse
ne suffit toutefois pas à expliquer toutes les erreurs :
l’exemple comportant 32,7 % de surface annotée obtient lui aussi
un score faible (0,041).

La superposition rouge représente l’annotation de référence,
et non les régions utilisées par le modèle pour sa décision.
Cette inspection ne constitue pas une validation médicale des
annotations ni une explication du fonctionnement interne du modèle.

L’analyse porte sur une seule lame avec micrométastase provenant
du centre 2. Elle ne permet pas de distinguer l’effet de
l’établissement, du type de lésion et des particularités de
cette lame. L’absence de lame avec micrométastase dans la
validation constitue également une limite du protocole.

Les paramètres restent inchangés après cette analyse du test.

Ce commentaire décrit les résultats historiques fournis, pas automatiquement les résultats d’une nouvelle exécution.


## 10. Conclusion

Les résultats et limites de l’expérience du 16 novembre sont présentés dans le notebook principal et le rapport actualisé. Ce fichier conserve le protocole de calcul ; il ne certifie pas une nouvelle exécution.